In [1]:
!pip install --quiet --upgrade ipywidgets langchain langchain-core tqdm pypdf pandas chromadb sentence-transformers 

In [2]:
import os
import re
import chromadb
import xml.etree.ElementTree as ET
import pandas as pd

from langchain_core.documents import Document
from nltk.tokenize import sent_tokenize
from pypdf import PdfReader
from dataclasses import dataclass
from typing import List
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# PDF-Extraktion

Die PDFs werden mithilfe von pypdf gelesen. Dazu gibt es eine Nachbearbeitung, damit die Zeilenumbrüche der PDF entfernt werden.

In [3]:
def load_sentences_from_pdfs_fraktionen(folder_path):
    pdf_sentences = {}
    
    pdf_files = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            pdf_files.append(file)

    for filename in pdf_files:
        filepath = os.path.join(folder_path, filename)

        reader = PdfReader(filepath)
        num_pages = len(reader.pages)

        sentences_in_file = []
        
        with tqdm(total=num_pages, desc=f"Verarbeite {filename}", unit="Seite") as pbar:
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    text = re.sub(r"^\s*\d+(\.\s*|\s+)", "", text, flags=re.MULTILINE)
                    
                    sentences = sent_tokenize(text)
                    sentences_in_file.extend(sentences)

                pbar.update(1)

        pdf_sentences[filename] = sentences_in_file

    return pdf_sentences

In [4]:
pdf_folder = "./BTW25_Downloads"
pdf_texts = {}

pdf_texts = load_sentences_from_pdfs_fraktionen(pdf_folder)

Verarbeite FDP_Wahlprogramm.pdf: 100%|██████████| 52/52 [00:02<00:00, 17.47Seite/s]


# XML-Parsing

Parser Klasse von Herr Gawron
https://github.com/fhswf/btp-crawler/tree/main

Die Parser Klasse wurde angepasst, damit zu jeder Rede auch der Sprecher mit der Partei gespeichert wird.

In [5]:
@dataclass
class SpeechItem:
    top_id: str
    speech_id: str
    speaker_id: str
    date: str
    type: str
    text: str
    party: str
    first_name: str
    last_name: str

In [6]:
class SpeechParser:

    attrib_class = 'klasse'
    class_speaker = 'redner'
    class_j1 = "J_1"
    tag_comment = 'kommentar'
    tag_name = 'name'

    @staticmethod
    def parse(xml_str: str) -> List[SpeechItem]:

        speech_date = None

        try:
            xml_root = ET.XML(SpeechParser.clean_xml(xml_str))
        except ET.ParseError as ex:
            print(ex)
            return []

        speech_date = xml_root.attrib.get('sitzung-datum')
        speech_items = []

        for topic in xml_root.iter('tagesordnungspunkt'):
            top_id = topic.attrib.get('top-id')

            for speech in topic.iter('rede'):
                
                redner_element = speech.find(".//redner")
                
                speech_id = speech.attrib.get('id')
                speaker_id = None
                skip_chairman = False
                speech_type = "rede"
                party = redner_element.findtext(".//fraktion", "n/a") # Redner die keine Fraktion zu gewiesen haben, bekommen "not applicable" zugewiesen, damit diese weiterhin auffindbar sind
                text = ""
                first_name = redner_element.findtext(".//vorname")
                last_name = redner_element.findtext(".//nachname")
                
                
                for line in speech:
                    if line.attrib.get(SpeechParser.attrib_class) in (SpeechParser.class_speaker, SpeechParser.tag_comment):
                        skip_chairman = False

                    if line.attrib.get(SpeechParser.attrib_class) == SpeechParser.class_speaker:
                        speaker_id = line.find(
                            SpeechParser.class_speaker).get('id')
                        continue

                    if line.tag == SpeechParser.tag_name:
                        skip_chairman = True

                    if skip_chairman:
                        continue

                    text += "".join(line.itertext())
                
                text = text.strip()
                if len(text) > 0: 
                    speech_items.append(
                        SpeechItem(
                            top_id,
                            speech_id,
                            speaker_id,
                            speech_date,
                            speech_type,
                            text,
                            party,
                            first_name,
                            last_name
                        )
                    )

        return speech_items

    @staticmethod
    def clean_xml(xml_str: str) -> str:
        # clean protected characters
        xml_str = \
            (xml_str.replace('\xa0', ' ')
                    .replace('\xad', '')
                    .replace('\u2011', '-')
                    .replace('\u2013', '-')
             )

        # remove line breaks and identing blanks
        if '\n' in xml_str:
            xml_str = re.sub('\n\s*', ' ', xml_str)

        return xml_str

In [7]:
folder_path = "./BTP20_Downloads/"

plenar_sitzungen = []

xml_files = []


for file in os.listdir(folder_path):
    if file.endswith(".xml"):
        xml_files.append(file)

for file_name in tqdm(xml_files, desc="Verarbeite Plenarsitzungenen", unit=" Plenarsitzungenen"):
    file_path = os.path.join(folder_path, file_name)
    
    if os.path.isfile(file_path):  
        with open(file_path, "r", encoding="utf-8") as xml:
            data = SpeechParser.parse(xml.read())  
            plenar_sitzungen += data  

print(f"Verarbeitung abgeschlossen. Gesamtzahl der Reden: {len(plenar_sitzungen)}")

Verarbeite Plenarsitzungenen: 100%|██████████| 208/208 [00:06<00:00, 31.37 Plenarsitzungenen/s]

Verarbeitung abgeschlossen. Gesamtzahl der Reden: 24671


In [8]:
df = pd.DataFrame(plenar_sitzungen)

In [9]:
# Verschiedene Partei Schreibweisen vereinheitlichen
df['party'] = df['party'].replace('Die Linke', 'DIE LINKE')
df['party'] = df['party'].replace('fraktionslos', 'Fraktionslos')

# Chunking der Texte

Die Wahlkapfprogramme und die Plenarsitzungen werden in kleine Chunks aufgeteilt, damit diese besser Embedded werden können.

In [10]:
def chunk_plenarsitzungen(df, text_splitter):
    chunks = []
    metadata_columns=["top_id", "speech_id", "speaker_id", "date", "type", "party", "first_name", "last_name"]
    text_column="text"
    
    # Prüfe ob alle Metadaten-Spalten existieren
    missing_columns = [col for col in metadata_columns if col not in df.columns]
    if missing_columns:
        print(f"Fehlende Spalten im DataFrame: {missing_columns}")

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Chunke Plenarsitzungen", unit="Rede"):
        text = row[text_column]
        
        metadata = {}
        for col in metadata_columns:
            if col in df.columns:
                value = row[col]
                metadata[col] = value if pd.notna(value) and value != "" else "unbekannt"
            else:
                metadata[col] = "unbekannt"
        
        for chunk in text_splitter.split_text(text):
            chunk_data = {"text": chunk, **metadata}
            chunks.append(chunk_data)

    return chunks

def chunk_wahlprogramme(pdf_texts, text_splitter):
    """
    Zerlegt die Texte aus `pdf_texts` in kleinere Chunks mit dem übergebenen `text_splitter`.

    :param pdf_texts: Dict mit Dateinamen als Keys und den zugehörigen Texten als Values (String oder Liste).
    :param text_splitter: Ein Text-Splitter-Objekt mit einer `split_text`-Methode.
    :return: Liste von Dictionaries mit Chunks der Wahlprogramme.
    """
    chunks = []

    for filename, text_value in tqdm(pdf_texts.items(), desc="Chunke Wahlprogramme", unit="Datei"):
        # Falls der Text als Liste vorliegt, zusammenfügen
        if isinstance(text_value, list):
            text_value = " ".join(text_value)

        # Text splitten und Chunks speichern
        for chunk in text_splitter.split_text(text_value):
            chunks.append({
                "text": chunk,
                "source": filename,
                "type": "wahlprogramm"
            })

    return chunks
    

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

wahlprogramme_chunks = []
plenarsitzungen_chunks = []


wahlprogramme_chunks = chunk_wahlprogramme(pdf_texts, text_splitter)

plenarsitzungen_chunks = chunk_plenarsitzungen(df, text_splitter)     

print(f"Wahlprogramme Chunks: {len(wahlprogramme_chunks)}")
print(f"Plenarsitzungen Chunks: {len(plenarsitzungen_chunks)}")

Chunke Plenarsitzungen: 100%|██████████| 24671/24671 [00:17<00:00, 1435.61Rede/s]

Wahlprogramme Chunks: 1685
Plenarsitzungen Chunks: 129079


# Embeddings erzeugen

In [12]:
!curl -d "Starte: Datenbank erstellen" ntfy.sh/JLoU0ayr9YME3Pwb

{"id":"zLicaOdVLyav","time":1738281778,"expires":1738324978,"event":"message","topic":"JLoU0ayr9YME3Pwb","message":"Starte: Datenbank erstellen"}


In [13]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

wahlprogramme_embeddings = []
plenarsitzungen_embeddings = []

# Wahlkampf-Embeddings generieren
for chunk in tqdm(wahlprogramme_chunks, desc="Wahlprogramme"):
    embedding = embedding_model.encode(
        chunk["text"], 
        convert_to_numpy=True,
        normalize_embeddings=True 
    )
        
    wahlprogramme_embeddings.append({
        "embedding": embedding,
        "text": chunk["text"],
        "source": chunk["source"],
        "type": "wahlprogramm"
    })

# Plenarsitzungen-Embeddings generieren
for chunk in tqdm(plenarsitzungen_chunks, desc="Plenarsitzungen"):
    embedding = embedding_model.encode(
        chunk["text"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    embedding_doc = {
        "embedding": embedding,
        "text": chunk["text"],
        "source": "bundestag",
        "type": "plenarsitzung",
        
        "speech_id": chunk["speech_id"],
        "top_id": chunk["top_id"],
        "speaker_id": chunk["speaker_id"],
        "date": chunk["date"],
        "party": chunk["party"],
        "first_name": chunk["first_name"],
        "last_name": chunk["last_name"]
    }
    
    plenarsitzungen_embeddings.append(embedding_doc)

print(f"Wahlprogramme-Embeddings: {len(wahlprogramme_embeddings)}")
print(f"Plenarsitzungen-Embeddings: {len(plenarsitzungen_embeddings)}")

Plenarsitzungen: 100%|██████████| 129079/129079 [1:29:22<00:00, 24.07it/s]

Wahlprogramme-Embeddings: 1685
Plenarsitzungen-Embeddings: 129079


# Speicherung in ChromaDB

In [14]:
chroma_client = chromadb.PersistentClient(path="./chromadb")

for collection_name in ["wahlprogramme", "plenarsitzungen"]:
    try:
        chroma_client.delete_collection(name=collection_name)
        print(f"Collection {collection_name} gelöscht")
    except Exception as e:
        print(f"Collection {collection_name} existierte nicht: {str(e)}")

wahlprogramme_collection = chroma_client.get_or_create_collection(name="wahlprogramme")
plenarsitzungen_collection = chroma_client.get_or_create_collection(name="plenarsitzungen")

# Speichern der Wahlprogramme-Embeddings
for entry in tqdm(wahlprogramme_embeddings, desc="Wahlprogramme in ChromaDB"):
    wahlprogramme_collection.add(
        documents=[entry["text"]],
        metadatas=[{"source": entry["source"], "type": entry["type"]}],
        ids=[entry["source"] + "_" + str(hash(entry["text"]))],
        embeddings=[entry["embedding"]]
    )

# Speichern der Plenarsitzungen-Embeddings
for entry in tqdm(plenarsitzungen_embeddings, desc="Plenarsitzungen in ChromaDB"):
    plenarsitzungen_collection.add(
        documents=[entry["text"]],
        metadatas=[{
            "top_id": entry.get("top_id", "n/a"),
            "speech_id": entry.get("speech_id", "n/a"),
            "speaker_id": entry.get("speaker_id", "n/a"),
            "date": entry.get("date", "n/a"),
            "type": entry.get("type", "n/a"),
            "party": entry.get("party", "n/a"),
            "first_name": entry.get("first_name", "n/a"),
            "last_name": entry.get("last_name", "n/a")
        }],
        ids=[entry["speech_id"] + "_" + str(hash(entry["text"]))],
        embeddings=[entry["embedding"]]
    )

print("Embeddings erfolgreich in ChromaDB gespeichert.")

print(f"Anzahl der gespeicherten Dokumente: {plenarsitzungen_collection.count()}")
print(f"Anzahl der gespeicherten Dokumente: {wahlprogramme_collection.count()}")

Collection wahlprogramme gelöscht
Collection plenarsitzungen gelöscht


Plenarsitzungen in ChromaDB: 100%|██████████| 129079/129079 [3:06:28<00:00, 11.54it/s]  


Embeddings erfolgreich in ChromaDB gespeichert.
Anzahl der gespeicherten Dokumente: 129079
Anzahl der gespeicherten Dokumente: 1685


In [15]:
!curl -d "Beendet: Datenbank erstellen" ntfy.sh/JLoU0ayr9YME3Pwb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{"id":"3tqM8oLm6SrW","time":1738298525,"expires":1738341725,"event":"message","topic":"JLoU0ayr9YME3Pwb","message":"Beendet: Datenbank erstellen"}


# Testausgaben

## Testausgabe aus ChromaDB

In [16]:
query = "Wie ist die Klimapolitik der Partei AFD?"
query_embedding = embedding_model.encode(query, convert_to_numpy=True)

results = wahlprogramme_collection.query(query_embeddings=[query_embedding], n_results=5)

print("Wahlprogramm-Ergebnisse:")
for doc in results["documents"][0]:
    print(doc)


Wahlprogramm-Ergebnisse:
Konzerne mit Ihren Lobbys und politikn ahe NGOs. Interessengruppen und ihre 1539 Bundesgeschäftsstelle der Partei Alternative für Deutschland | Eichhorster Weg 80 | 13435 Berlin 
unterstützenden Parteien schaffen so zunehmend Tätigkeitsfelder für ihre eigene 1540 
Klientel – ohne jede Wertschöpfung. 1541 
Die AfD lehnt daher jede Politik und jede Steuer ab, die sich auf angeblichen Klimaschutz 1542 
beruft, denn das Klima kann der Mensch nicht schützen. Wir wollen zudem aus dem 1543 
Pariser Klimaabkommen aussteigen. 1544 
Die AfD wird unseren zukünftigen Generationen die Hoffnung und die Möglichkeit auf 1545 
ein würdiges Leben in Freiheit und Wohlstan d zurückbringen. Die ausufernde Plan- und 1546 
Subventionswirtschaft der letzten Jahrzehn te werden wir in eine moderne soziale 1547 
Marktwirtschaft zurückführen, mit der wir alle kommenden Herausforderungen 1548 
meistern können. Es ist noch nicht zu sp ät, die von linksgrünen Ideologen zerstörte 1549
Die AfD

In [17]:
query = "Umwelt"
query_embedding = embedding_model.encode(query, convert_to_numpy=True)

results = plenarsitzungen_collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"party": "AfD"}  # Filter nach Partei
)

print("Ergebnisse:")
for doc in results["documents"][0]:
    print(doc)

Ergebnisse:
Anpassung an den Klimawandel, kooperative Ansätze beim Artenschutz gemeinsam mit der Land- und Forstwirtschaft sowie bei der Bekämpfung -- von invasiven, gebietsfremden Arten dringend benötigt.Vielen Dank.(Beifall bei der AfD)
anschaffen, die anstelle von Flugzeugen zur Anreise genutzt werden?(Beifall bei der AfD)Was meinen Sie überhaupt mit dem Beitrag des Fußballs zu mehr Nachhaltigkeit? Was haben Umweltverbände mit Sportverbänden zu tun?(Dr. Jan-Niclas Gesenhues [BÜNDNIS 90/DIE GRÜNEN]: Das haben Sie noch gar nicht kapiert?)Das Bundesumweltministerium sollte sich eigentlich ganz anderen Fragen zuwenden als der nächsten EM, zum Beispiel der Frage, warum die EU die Kernkraft für nachhaltig hält, Deutschland aber wieder mal einen Sonderweg gehen will,(Beifall bei der AfD)oder der Frage, warum uns die schwarz-rot-grün-gelbe Altparteienriege durch ihren kopflosen Kernkraftausstieg innerhalb von zehn Jahren völlig in die Abhängigkeit von Russland manövriert hat.(Filiz Polat [B

In [18]:
query = "Umwelt"
query_embedding = embedding_model.encode(query, convert_to_numpy=True)

results = plenarsitzungen_collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"party": "BÜNDNIS 90/DIE GRÜNEN"}  # Filter nach Partei
)

print("Ergebnisse:")
for doc in results["documents"][0]:
    print(doc)
    
    

Ergebnisse:
Klimaschutz in Einklang zu bringen. Jüngst hat eine renommierte Studie gezeigt, dass sechs von neun planetaren Grenzen überschritten sind, zuvorderst bei der Artenvielfalt.Fazit: Wir werden Mutter Erdes warmen Kern umweltverträglich und zügig und mit Augenmaß für unsere Wärmeversorgung nutzen.Vielen Dank.(Beifall beim BÜNDNIS 90/DIE GRÜNEN und bei der SPD sowie bei Abgeordneten der FDP)
wir ein. Der Erhalt der Biodiversität und der Schutz des Klimas sind die elementaren Grundlagen für unser Leben und unser Wirtschaften.Herzlichen Dank.(Beifall beim BÜNDNIS 90/DIE GRÜNEN und bei der SPD sowie bei Abgeordneten der FDP)
der Konferenz in Montreal haben sich nicht nur Vertreterinnen und Vertreter von Wissenschaft, Politik und Zivilgesellschaft in der Frankfurter Erklärung für natur-positives unternehmerisches Handeln ausgesprochen. Auch die Wirtschaft war dabei prominent vertreten.Der Weltbiodiversitätsrat beziffert in einer gestern vorgestellten Studie alleine den Schaden, der 

## Dataframe

In [19]:
df

,top_id,speech_id,speaker_id,date,type,text,party,first_name,last_name
0,Tagesordnungspunkt 1,ID2018300100,11004097,10.09.2024,rede,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,n/a,Christian,Lindner
1,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300200,11004110,10.09.2024,rede,Liebe Frau Präsidentin! Liebe Kolleginnen! Lie...,CDU/CSU,Mathias,Middelberg
2,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300300,11004388,10.09.2024,rede,Geschätzte Frau Präsidentin! Liebe Kolleginnen...,SPD,Dennis,Rohde
3,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300400,11004675,10.09.2024,rede,Frau Präsidentin! Eines muss man vor die Klamm...,AfD,Peter,Boehringer
4,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300500,11004070,10.09.2024,rede,Sehr geehrte Frau Präsidentin! Sehr geehrter H...,BÜNDNIS 90/DIE GRÜNEN,Sven-Christian,Kindler
...,...,...,...,...,...,...,...,...,...
24666,Tagesordnungspunkt 18,ID2016316100,11005002,11.04.2024,rede,Danke schön. - Frau Präsidentin! Liebe Kollegi...,CDU/CSU,Knut,Abraham
24667,Tagesordnungspunkt 18,ID2016316200,11005229,11.04.2024,rede,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,SPD,Ralf,Stegner
24668,Tagesordnungspunkt 18,ID2016316300,11005264,11.04.2024,rede,Frau Präsidentin! Meine Damen und Herren! Herr...,AfD,Joachim,Wundrak
24669,Tagesordnungspunkt 18,ID2016316400,11005226,11.04.2024,rede,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,BÜNDNIS 90/DIE GRÜNEN,Merle,Spellerberg


In [20]:
party_dataframes = {party: group for party, group in df.groupby('party')}

In [21]:
party_dataframes["AfD"]

,top_id,speech_id,speaker_id,date,type,text,party,first_name,last_name
3,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300400,11004675,10.09.2024,rede,Frau Präsidentin! Eines muss man vor die Klamm...,AfD,Peter,Boehringer
8,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300900,11004932,10.09.2024,rede,Frau Präsidentin! Sehr verehrte Kollegen von d...,AfD,Harald,Weyel
22,Einzelplan 11,ID2018302300,11004781,10.09.2024,rede,Werter Herr Präsident! Sehr geehrte Kolleginne...,AfD,Norbert,Kleinwächter
26,Einzelplan 11,ID2018302700,11004873,10.09.2024,rede,Herr Präsident! Liebe Kollegen! Verehrte Bürge...,AfD,Ulrike,Schielke-Ziesing
41,Einzelplan 10,ID2018304200,11004714,10.09.2024,rede,Frau Präsidentin! Herr Minister! Meine Damen u...,AfD,Peter,Felser
...,...,...,...,...,...,...,...,...,...
24643,Tagesordnungspunkt 15,ID2016313800,11004707,11.04.2024,rede,Frau Präsidentin! Meine Damen und Herren! Natü...,AfD,Thomas,Ehrhorn
24650,Tagesordnungspunkt 14,ID2016314500,11004858,11.04.2024,rede,Sehr geehrte Frau Präsidentin! Meine Damen und...,AfD,Stephan,Protschka
24659,Tagesordnungspunkt 16,ID2016315400,11004836,11.04.2024,rede,Sehr geehrte Frau Präsidentin! Meine Damen und...,AfD,Sebastian,Münzenmaier
24663,Tagesordnungspunkt 17,ID2016315800,11005100,11.04.2024,rede,Frau Präsidentin! Geehrte Kollegen! Das hier d...,AfD,Michael,Kaufmann


In [22]:
party_dataframes["CDU/CSU"]

,top_id,speech_id,speaker_id,date,type,text,party,first_name,last_name
1,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300200,11004110,10.09.2024,rede,Liebe Frau Präsidentin! Liebe Kolleginnen! Lie...,CDU/CSU,Mathias,Middelberg
6,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018300700,11004682,10.09.2024,rede,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,CDU/CSU,Sebastian,Brehm
11,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018301200,11004286,10.09.2024,rede,Sehr geehrter Herr Präsident! Herr Finanzminis...,CDU/CSU,Christian,Haase
16,"Allgemeine Finanzdebatte, Einzelplan 08",ID2018301700,11003646,10.09.2024,rede,Herr Präsident! Liebe Kolleginnen und Kollegen...,CDU/CSU,Antje,Tillmann
20,Einzelplan 11,ID2018302100,11002666,10.09.2024,rede,Herr Präsident! Herr Minister! Liebe Kolleginn...,CDU/CSU,Hermann,Gröhe
...,...,...,...,...,...,...,...,...,...
24660,Tagesordnungspunkt 16,ID2016315500,11004741,11.04.2024,rede,Frau Präsidentin! Meine Damen und Herren! Beim...,CDU/CSU,Thomas,Heilmann
24662,Tagesordnungspunkt 17,ID2016315700,11004241,11.04.2024,rede,Liebe Frau Präsidentin! Liebe Kolleginnen und ...,CDU/CSU,Stephan,Albani
24665,Tagesordnungspunkt 17,ID2016316000,11004116,11.04.2024,rede,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...,CDU/CSU,Nadine,Schön
24666,Tagesordnungspunkt 18,ID2016316100,11005002,11.04.2024,rede,Danke schön. - Frau Präsidentin! Liebe Kollegi...,CDU/CSU,Knut,Abraham


In [23]:
for party, party_df in party_dataframes.items():
    print(f"{party}:{len(party_df)}")

AfD:2956
BSW:138
BÜNDNIS 90/DIE GRÜNEN:3231
CDU/CSU:5419
DIE LINKE:1703
FDP:2594
Fraktionslos:444
SPD:4694
SPDCDU/CSU:6
n/a:3486


In [24]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

wahlprogramme_embeddings = []
plenarsitzungen_embeddings = []

max_chunks = 1000

# Wahlkampf-Embeddings generieren
for chunk in tqdm(wahlprogramme_chunks[:max_chunks], desc="Wahlprogramme"):
    embedding = embedding_model.encode(
        chunk["text"], 
        convert_to_numpy=True,
        normalize_embeddings=True 
    )
        
    wahlprogramme_embeddings.append({
        "embedding": embedding,
        "text": chunk["text"],
        "source": chunk["source"],
        "type": "wahlprogramm"
    })

# Plenarsitzungen-Embeddings generieren
for chunk in tqdm(plenarsitzungen_chunks[:max_chunks], desc="Plenarsitzungen"):
    embedding = embedding_model.encode(
        chunk["text"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    embedding_doc = {
        "embedding": embedding,
        "text": chunk["text"],
        "source": "bundestag",
        "type": "plenarsitzung",
        
        "speech_id": chunk["speech_id"],
        "top_id": chunk["top_id"],
        "speaker_id": chunk["speaker_id"],
        "date": chunk["date"],
        "party": chunk["party"],
        "first_name": chunk["first_name"],
        "last_name": chunk["last_name"]
    }
    
    plenarsitzungen_embeddings.append(embedding_doc)

print(f"Wahlprogramme-Embeddings: {len(wahlprogramme_embeddings)}")
print(f"Plenarsitzungen-Embeddings: {len(plenarsitzungen_embeddings)}")

Plenarsitzungen: 100%|██████████| 1000/1000 [00:42<00:00, 23.60it/s]

Wahlprogramme-Embeddings: 1000
Plenarsitzungen-Embeddings: 1000
